# 08 · The tools the AI calls

**Purpose.** Walk through `src/indurent_matching/tools.py`: the functions the model will call to answer requests.
Each is called here exactly as the model would call it, and the result is shown as the dictionary the model
receives (pretty-printed JSON), with a table alongside where that's easier to read.

The tools are plain Python, with no AI involved: they wrap the similarity functions (notebooks 05-07) and return a
short `summary`, the rows that matter, and `caveats` the answer should pass on. The model never sees the whole
table.

In [1]:
import json
from pathlib import Path

import pandas as pd

from indurent_matching.data import load_marketed, load_portfolio
from indurent_matching.tools import (
    Data,
    ToolError,
    filter_rows,
    find_nearest,
    find_similar,
    get_homogeneity_scores,
    get_listing,
    list_listings,
)


ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
data = Data(load_portfolio(ROOT / "data" / "Current Portfolio.xlsx"),
            load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx"))


def show(result):
    """Print a tool result as the model receives it."""
    print(json.dumps(result, indent=2))

`Data` holds the two cleaned tables. Every tool takes it as its first argument; in the agent (`graph.py`) that's
filled in automatically, so the model only supplies the other arguments.

## 1. `list_listings`: what can be compared

Usually the first call when a request doesn't name a listing.

In [2]:
result = list_listings(data)
print(result["summary"])
pd.DataFrame(result["listings"]).set_index("listing_id")[
    ["estate", "unit", "size_sqft", "build_year", "location_confidence", "matched_on", "not_available"]]

5 marketed listings: M1, M2, M3, M4, M5


,estate,unit,size_sqft,build_year,location_confidence,matched_on,not_available
listing_id,,,,,,,
M1,Cherry Lane,Unit 10,16860,1977.0,medium,"[age, location, physical]",{}
M2,Tech Hub,Unit 8,906,NaN,unknown,[physical],"{'age': 'no build year', 'location': 'no coord..."
M3,Spitfire Park,Unit 42,2294,1976.0,high,"[age, location, physical]",{}
M4,Stable Lane,Unit 14,617,2000.0,medium,"[age, location, physical]",{}
M5,Chancery Depot,Unit 2,16614,1966.0,unknown,"[age, physical]","{'location': 'coordinates are offshore, so can..."


* All five listings, with what each can be matched on and why not: **M2** has no build year or coordinates, so
  it can only be matched on size; **M5**'s coordinates are offshore.

## 2. `get_listing`: resolve which property the user means

By id or by name, forgiving about case and partial names:

In [3]:
show(get_listing(data, "spitfire park"))

{
  "summary": "M3: Spitfire Park, Unit 42",
  "listing_id": "M3",
  "estate": "Spitfire Park",
  "unit": "Unit 42",
  "region_label": "Midlands",
  "size_sqft": 2294,
  "build_year": 1976,
  "location_confidence": "high",
  "nearest_city": "Manchester",
  "miles_to_city": 14.0,
  "epc_rating": "B",
  "matched_on": [
    "age",
    "location",
    "physical"
  ],
  "not_available": {},
  "sits_on_estate": "Velvet Park 60",
  "quality_notes": []
}


In [4]:
for query in ["M9", "Unit 1"]:
    try:
        get_listing(data, query)
    except ToolError as error:
        print(f"get_listing({query!r}) -> ToolError: {error}")

get_listing('M9') -> ToolError: No marketed listing matches 'M9'. The listings are: M1 (Cherry Lane, Unit 10), M2 (Tech Hub, Unit 8), M3 (Spitfire Park, Unit 42), M4 (Stable Lane, Unit 14), M5 (Chancery Depot, Unit 2)
get_listing('Unit 1') -> ToolError: 'Unit 1' matches more than one listing; use one of: M1 (Cherry Lane, Unit 10), M4 (Stable Lane, Unit 14)


* "spitfire park" finds M3, with its facts, the estate it sits on (Velvet Park 60) and its quality notes (none).
* Bad requests raise a **`ToolError` written for the model to read and correct**: an unknown listing lists the
  options; an ambiguous one ("Unit 1" matches Unit 10 and Unit 14) asks for one of the matches.

## 3. `filter_rows`: narrow the portfolio (prompt 3)

> "Find the closest properties to the marketed property, after excluding any property in the portfolio that is
> more than 10 miles from a major city."

"Excluding any property more than 10 miles away" means **keep `miles_to_city <= 10`**. `include_unknown` must be
given: it decides what happens to the units whose location can't be trusted. Both choices:

In [5]:
within_10 = [{"field": "miles_to_city", "op": "<=", "value": 10}]
excluded = filter_rows(data, within_10, include_unknown=False)
kept = filter_rows(data, within_10, include_unknown=True)
print(excluded["summary"])
print(kept["summary"])
print(f"\ncandidates (first 10 of {len(excluded['candidates'])}): {excluded['candidates'][:10]}")

Kept 50 of 1,250 units (miles_to_city <= 10). 358 had no usable value for miles_to_city and were excluded.
Kept 408 of 1,250 units (miles_to_city <= 10). 358 had no usable value for miles_to_city and were kept.

candidates (first 10 of 50): [167, 168, 169, 170, 275, 276, 277, 278, 336, 337]


* **50 of 1,250** units are known to be within 10 miles of a major city. **358 have no usable location**: excluded,
  the comparison is against 50 units; kept, against 408. The summary always reports the 358, so the answer can say
  which choice was made.
* The result's `candidates` (Property Ids) are the new set; the agent remembers them and passes them to the next
  tool.

Filters chain, and conditions combine (all must hold). For example, the units within 10 miles that are also in the
Midlands and over 10,000 sq ft:

In [6]:
midlands_big = filter_rows(
    data,
    [{"field": "derived_region", "op": "==", "value": "Midlands"}, {"field": "size_sqft", "op": ">", "value": 10_000}],
    include_unknown=False,
    candidates=excluded["candidates"],
)
print(midlands_big["summary"])

Kept 9 of 50 units (derived_region == Midlands and size_sqft > 10000). 0 had no usable value for derived_region, size_sqft and were excluded.


In [7]:
for bad in [{"field": "eaves", "op": "<", "value": 5}, {"field": "derived_region", "op": "<", "value": "M"}]:
    try:
        filter_rows(data, [bad], include_unknown=False)
    except ToolError as error:
        print(f"{bad} -> ToolError: {error}")

{'field': 'eaves', 'op': '<', 'value': 5} -> ToolError: Can't filter on 'eaves'. Fields: ['build_year', 'car_parking_spaces', 'derived_region', 'epc_rating', 'estate_units', 'location_confidence', 'miles_to_city', 'nearest_city', 'size_sqft']
{'field': 'derived_region', 'op': '<', 'value': 'M'} -> ToolError: '<' doesn't apply to derived_region (text). Use one of: ['==', 'in']


* The second filter started from the 50 and kept the ones meeting both new conditions.
* Fields and operators are a fixed list, and a mistake comes back as a message the model can fix.

## 4. `find_similar`: the most similar units (prompt 1)

Wraps `most_similar`. The matches, as a table, then the rest of what the model receives:

In [8]:
result = find_similar(data, "M3")
print(result["summary"])
pd.DataFrame(result["matches"]).set_index("property_id")

Top 10 matches for M3 (Spitfire Park, Unit 42) from 1,250 units


,estate,unit,overall,age,location,physical,build_year,size_sqft,miles_from_listing,doubtful_fields
property_id,,,,,,,,,,
869,Velvet Park 60,Unit 24,0.840,0.812,0.993,0.714,1973,1638,0.1,[]
865,Velvet Park 60,Unit 20,0.815,0.758,0.998,0.689,1980,3332,0.0,[]
863,Velvet Park 60,Unit 18,0.799,1.000,0.996,0.402,1976,923,0.1,[]
855,Velvet Park 60,Unit 10,0.797,0.871,0.996,0.523,1978,1199,0.1,[]
878,Velvet Park 60,Unit 33,0.772,0.660,0.993,0.663,1970,1520,0.1,[]
849,Velvet Park 60,Unit 4,0.771,0.435,0.999,0.877,1988,2013,0.0,[]
851,Velvet Park 60,Unit 6,0.740,0.933,0.994,0.294,1977,675,0.1,[]
860,Velvet Park 60,Unit 15,0.734,0.758,0.994,0.451,1980,5086,0.1,[]
852,Velvet Park 60,Unit 7,0.732,0.406,0.999,0.790,1989,2904,0.0,[]


In [9]:
show({key: value for key, value in result.items() if key != "matches"})

{
  "summary": "Top 10 matches for M3 (Spitfire Park, Unit 42) from 1,250 units",
  "listing": {
    "listing_id": "M3",
    "estate": "Spitfire Park",
    "unit": "Unit 42",
    "region_label": "Midlands",
    "size_sqft": 2294,
    "build_year": 1976,
    "location_confidence": "high",
    "nearest_city": "Manchester",
    "miles_to_city": 14.0,
    "epc_rating": "B",
    "matched_on": [
      "age",
      "location",
      "physical"
    ],
    "not_available": {}
  },
  "caveats": [
    "The listing sits on Velvet Park 60, an estate Indurent already owns; 10 of the 10 matches are on it."
  ]
}


* M3's ten are all on Velvet Park 60, and a caveat says why: M3 sits on that estate, which Indurent already owns.
* `listing` repeats the listing's facts, so the answer can describe it without another call.

**When a listing is missing data**, the caveats carry it. M2 can only be matched on size:

In [10]:
for caveat in find_similar(data, "M2")["caveats"]:
    print("-", caveat)

- Matched on physical only. Not available: age (no build year); location (no coordinates).
- 11 units score 0.98 or more on size alone, so the list is one of several equally good ones.


## 5. `get_homogeneity_scores`: how typical the listing is (prompt 2)

In [11]:
show(get_homogeneity_scores(data, "M5"))

{
  "summary": "Homogeneity scores for M5 (Chancery Depot, Unit 2)",
  "meaning": "Homogeneity here means how typical the listing is of Indurent's portfolio, scored 0-1 on each dimension, where 1 is the most typical value in the portfolio. 'Correlation score' in the request is read this way.",
  "scores": {
    "age": 0.2,
    "location": null,
    "physical": 0.41
  },
  "not_available": {
    "location": "coordinates are offshore, so can't be trusted"
  },
  "location_context": {
    "sits_on_estate": null,
    "nearest_other_estate": null,
    "nearest_other_estate_miles": null,
    "better_placed_than_pct": null
  },
  "size_context": {
    "eaves_m": 5.3,
    "similar_size_units_with_eaves": 45,
    "their_median_eaves_m": 6.3,
    "eaves_higher_than_pct": 15.6
  },
  "listing": {
    "listing_id": "M5",
    "estate": "Chancery Depot",
    "unit": "Unit 2",
    "region_label": null,
    "size_sqft": 16614,
    "build_year": 1966,
    "location_confidence": "unknown",
    "nearest_

* The three scores, with **null for not available and the reason** (M5's location: its coordinates are offshore).
* `meaning` gives the interpretation of "correlation score" for the answer to state.
* `size_context` gives M5's eaves against similar-sized units (5.3 m against a median of 6.3 m).

## 6. `find_nearest`: the nearest units on the map (prompt 3)

Prompt 3 asks for the **closest** properties, which we read as **nearest on the map**: a different question from
prompt 1's "most similar". `find_nearest` ranks units by straight-line miles. Prompt 3 end to end is two calls:
`filter_rows` (above) to keep the 50 units within 10 miles of a major city, then `find_nearest` on that set:

In [12]:
result = find_nearest(data, "M3", candidates=excluded["candidates"])
print(result["summary"])
for caveat in result["caveats"]:
    print("-", caveat)
pd.DataFrame(result["matches"]).set_index("property_id")

The 10 units nearest to M3 (Spitfire Park, Unit 42) on the map, from 50 units with a usable location
- The listing sits on Velvet Park 60, an estate Indurent already owns; 0 of the 10 nearest are on it.
- Searched a filtered set of 50 units, not the whole portfolio.


,estate,unit,miles_from_listing,build_year,size_sqft,nearest_city,miles_to_city
property_id,,,,,,,
170,Jade Bridge 6,Unit 4,9.1,1989,7908,Manchester,7.1
169,Jade Bridge 6,Unit 3,9.1,1958,17520,Manchester,7.1
167,Jade Bridge 6,Unit 1,9.1,1977,7542,Manchester,7.1
168,Jade Bridge 6,Unit 2,9.2,1968,8239,Manchester,7.1
503,Scarlet Arc 73,Unit 1,16.2,2005,134579,Sheffield,9.0
1162,Topaz Harbour 22,Unit 1,51.5,2004,15268,Nottingham,8.2
483,Onyx Oaks 5,Unit 1,62.7,1987,173913,Leicester,9.0
348,Marble Works 29,Unit 1,64.8,1987,887038,Birmingham,7.9
401,Sapphire Junction 78,Unit 4,68.8,2000,38568,Birmingham,8.7


* **M3's nearest unit within 10 miles of a city is Jade Bridge 6, 9.1 miles away.** None of Velvet Park 60, the
  estate M3 sits on, survives the filter: it's 14 miles from Manchester, and the caveat says "0 of the 10 nearest
  are on it".
* The caveats record that the search was over a filtered set of 50, so the answer can say so (and, from the
  filter's summary, that 358 units with no usable location were left out).

A listing that can't be placed on the map has no nearest properties; the error says why, for the answer to pass
on:

In [13]:
for listing in ["M2", "M5"]:
    try:
        find_nearest(data, listing)
    except ToolError as error:
        print(f"find_nearest({listing!r}) -> ToolError: {error}")

find_nearest('M2') -> ToolError: M2 can't be placed on the map: no coordinates. Nearest properties can't be found for it; say so in the answer.
find_nearest('M5') -> ToolError: M5 can't be placed on the map: coordinates are offshore, so can't be trusted. Nearest properties can't be found for it; say so in the answer.


## 7. Summary

* Each tool returns a short `summary`, the facts or rows that matter, and `caveats`; never the whole table.
* The filtered set is passed in and out as Property Ids, so prompt 3 is two calls: `filter_rows`, then
  `find_nearest` with `candidates`.
* The next step, `graph.py`, gives these tools to the model, keeps the candidate set between calls, and checks
  that every Property Id in an answer came from a tool.